## JobSpy Job Scraper — Munich Working Student (Data Scientist / Analyst) Roles

This notebook is **self-healing and self-resolving**: it installs JobSpy, patches a
known Glassdoor bug, then **automatically tries multiple location and search-term
variants per platform** until it finds one that returns results — so you don't have
to manually guess the exact string each job board expects.

**What it does:**
1. Installs/upgrades `python-jobspy` (with fallback for externally-managed Python)
2. Monkey-patches the Glassdoor location-lookup bug at runtime (no manual file editing)
3. **Auto-resolves location + search term per platform** — tries a list of candidate
   variants (e.g. `["München", "Bayern"]`, `["Munich", "Bavaria"]`) and keeps the
   first one that returns at least one job
4. Runs Google Jobs separately, since it requires a different query format entirely
5. Merges everything into one DataFrame with a guaranteed `description` column
6. Saves results to `jobs.csv`


## Python Installs and Fixes

In [1]:
# Install / upgrade JobSpy. Falls back automatically on "externally managed
# environment" errors (common on some system Python installs on Linux/Mac;
# not an issue inside a normal Windows venv).
import subprocess, sys

def pip_install(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        print("System Python is externally managed — retrying with --break-system-packages")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "--break-system-packages", *args],
            capture_output=True, text=True,
        )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"pip install failed for: {args}")
    installed_names = " ".join(args)
    print(f"Installed/updated: {installed_names}")

pip_install("python-jobspy")


Installed/updated: python-jobspy


#### Auto-patch the Glassdoor location bug

As of `python-jobspy` 1.1.82 (and the unmerged GitHub PR #347), the Glassdoor
scraper builds its location-lookup URL with an f-string and **never
URL-encodes the location string**:

```python
url = f"{self.base_url}/findPopularLocationAjax.htm?maxLocationsToReturn=10&term={location}"
```

Any location with a comma, space, or non-ASCII character (e.g. `"München, Bayern"`)
gets sent raw, which Glassdoor's endpoint rejects with HTTP 400 → `"location not parsed"`.

Rather than editing the installed package file by hand (which breaks on the next
`pip install`/on a different machine), the cell below **monkey-patches the method
at runtime, in memory**, every time the notebook runs. This requires no
filesystem edits and survives reinstalls/upgrades/other machines.


In [2]:
import inspect
import urllib.parse
import jobspy.glassdoor as gd_module

def _patched_get_location(self, location: str, is_remote: bool):
    """Drop-in replacement for Glassdoor._get_location with URL-encoding fixed."""
    if not location or is_remote:
        return "11047", "STATE"  # remote options
    term = urllib.parse.quote(location)
    url = f"{self.base_url}/findPopularLocationAjax.htm?maxLocationsToReturn=10&term={term}"
    res = self.session.get(url)
    if res.status_code != 200:
        if res.status_code == 429:
            err = "429 Response - Blocked by Glassdoor for too many requests"
        else:
            err = f"Glassdoor response status code {res.status_code}"
        gd_module.log.error(err)
        return None, None
    items = res.json()
    if not items:
        raise ValueError(f"Location \'{location}\' not found on Glassdoor")
    location_type = items[0]["locationType"]
    if location_type == "C":
        location_type = "CITY"
    elif location_type == "S":
        location_type = "STATE"
    elif location_type == "N":
        location_type = "COUNTRY"
    return int(items[0]["locationId"]), location_type

_patched_get_location._is_jobspy_location_patch = True


def patch_glassdoor_location_bug():
    """Apply the runtime patch only if the installed version still has the bug.
    Safe to call multiple times (idempotent) and safe once JobSpy ships a real fix
    upstream (it self-detects and skips)."""
    current = gd_module.Glassdoor._get_location
    if getattr(current, "_is_jobspy_location_patch", False):
        print("Patch already applied this session — skipping.")
        return False
    try:
        source = inspect.getsource(current)
        if "quote(" in source:
            print("JobSpy already fixed upstream — no patch needed.")
            return False
    except OSError:
        pass  # can\'t introspect source (e.g. already patched in a prior cell run) — proceed
    gd_module.Glassdoor._get_location = _patched_get_location
    print("Patched Glassdoor._get_location (URL-encoding fix applied).")
    return True

patch_glassdoor_location_bug()


Patched Glassdoor._get_location (URL-encoding fix applied).


True

## Search - Config
Different job boards index locations and parse search terms differently — what
works on Glassdoor (`"München, Bayern"`) may not work on Indeed, and vice versa.
Rather than hardcoding one string and hoping, define an **ordered list of
candidate variants** per category. The resolver below tries each in order and
keeps the first one that returns at least one job, for each site independently.

Edit these lists for a different city/role/country — everything downstream
adapts automatically.


In [ ]:
# Ordered candidate variants — first one that returns results wins, per site.
# Add/reorder freely; put your best guess first to minimize wasted requests.

LOCATION_VARIANTS = [
    "München, Bayern",
    "Munich, Bavaria",
    "München",
    "Munich",
    "Munich, Germany",
    "Bayern",
    "Bavaria, Germany",
]

SEARCH_TERM_VARIANTS = [
    "Werkstudent Data Scientist Analyst",
    "Werkstudent Data Science",
    '"working student" (data scientist OR analyst)',
    "working student data analyst",
    "Werkstudent Data", 
    "werkstudent ai", 
    "werkstudent ml",
    "werkstudent ki",
    "working student ai",
    "working student ki",
    "working student ml",
    "Werkstudent ML",
    "Werkstudent AI",
    "Werkstudent KI",
    "working Student AI",
    "working student KI",
    "working student ML",
]

# Google Jobs ignores search_term/location entirely — it needs one natural-language
# query with the location and timeframe written into the sentence itself.
GOOGLE_SEARCH_TERM_VARIANTS = [
    "working student data scientist or analyst jobs in Munich, Germany since yesterday",
    "Werkstudent data scientist jobs near Munich, Germany",
    "data analyst working student jobs in Munich Germany",
]

SITES_TO_TRY = ["glassdoor", "indeed", "linkedin"]  # add "zip_recruiter" if desired
COUNTRY_INDEED = "Germany"
RESULTS_WANTED = 200
HOURS_OLD = 7200

# LinkedIn only populates the `description` column when this is True — otherwise
# it's always NaN. Fetching descriptions means JobSpy visits each job's page
# individually, so it's noticeably slower (and more likely to get rate-limited)
# than the default fast pass. Set False if you don't need LinkedIn descriptions
# and want speed instead.
LINKEDIN_FETCH_DESCRIPTION = True

# Round-robin proxy list, format: "user:pass@host:port" or "host:port" or "localhost".
# Leave as None to make requests directly without a proxy.
PROXIES = None  # e.g. ["208.195.175.46:65095", "208.195.175.45:65095", "localhost"]


## Per-site auto-resolver

For each site, this tries every `(location, search_term)` combination in order
(location outer loop, search term inner loop) and stops at the first combination
that returns at least one job. Each site gets resolved independently, since
Glassdoor and Indeed may need different variants to succeed.

`LINKEDIN_FETCH_DESCRIPTION` and `PROXIES` (set in the config cell above) are
passed through to every call here. LinkedIn descriptions specifically require
JobSpy to visit each job's page individually after the search — noticeably
slower than the other sites and more likely to trigger rate limiting, since
LinkedIn is the most aggressive blocker among supported boards.

This makes real network requests — expect it to take a little while, especially
if early variants fail and it has to retry, or if `LINKEDIN_FETCH_DESCRIPTION`
is on.


In [22]:
from jobspy import scrape_jobs
import pandas as pd

def resolve_and_scrape(site, location_variants, search_term_variants,
                        results_wanted=RESULTS_WANTED, hours_old=72, country_indeed=COUNTRY_INDEED,
                        linkedin_fetch_description=False, proxies=None,
                        verbose=1):
    """Try (location, search_term) combinations for one site until one works.
    Returns (dataframe, location_used, search_term_used) — dataframe is empty
    if every combination failed."""
    for location in location_variants:
        for search_term in search_term_variants:
            try:
                df = scrape_jobs(
                    site_name=[site],
                    search_term=search_term,
                    location=location,
                    results_wanted=results_wanted,
                    hours_old=hours_old,
                    country_indeed=country_indeed,
                    linkedin_fetch_description=linkedin_fetch_description,
                    proxies=proxies,
                    verbose=verbose,
                )
            except Exception as e:
                print(f"  [{site}] location={location!r} search_term={search_term!r} -> raised {type(e).__name__}: {e}")
                continue
            if df is not None and not df.empty:
                print(f"  [{site}] SUCCESS with location={location!r} search_term={search_term!r} -> {len(df)} jobs")
                return df, location, search_term
            else:
                print(f"  [{site}] location={location!r} search_term={search_term!r} -> 0 jobs, trying next variant")
    print(f"  [{site}] All variants exhausted — no results found.")
    return pd.DataFrame(), None, None


results = {}
for site in SITES_TO_TRY:
    print(f"Resolving site: {site}")
    df, used_location, used_term = resolve_and_scrape(
        site,
        LOCATION_VARIANTS,
        SEARCH_TERM_VARIANTS,
        results_wanted=RESULTS_WANTED,
        hours_old=HOURS_OLD,
        country_indeed=COUNTRY_INDEED,
        linkedin_fetch_description=LINKEDIN_FETCH_DESCRIPTION,
        proxies=PROXIES,
    )
    results[site] = {"df": df, "location": used_location, "search_term": used_term}

for site, info in results.items():
    n = len(info["df"])
    used_loc = info["location"]
    used_term = info["search_term"]
    print(f"{site}: {n} jobs (location={used_loc!r}, search_term={used_term!r})")


Resolving site: glassdoor
  [glassdoor] location='München, Bayern' search_term='Werkstudent Data Scientist Analyst' -> 0 jobs, trying next variant
  [glassdoor] SUCCESS with location='München, Bayern' search_term='Werkstudent Data Science' -> 38 jobs
Resolving site: indeed
  [indeed] location='München, Bayern' search_term='Werkstudent Data Scientist Analyst' -> 0 jobs, trying next variant
  [indeed] SUCCESS with location='München, Bayern' search_term='Werkstudent Data Science' -> 50 jobs
Resolving site: linkedin
  [linkedin] SUCCESS with location='München, Bayern' search_term='Werkstudent Data Scientist Analyst' -> 57 jobs
glassdoor: 38 jobs (location='München, Bayern', search_term='Werkstudent Data Science')
indeed: 50 jobs (location='München, Bayern', search_term='Werkstudent Data Science')
linkedin: 57 jobs (location='München, Bayern', search_term='Werkstudent Data Scientist Analyst')


### Resolve Google

In [23]:
def resolve_google(google_search_term_variants, results_wanted=20, proxies=None, verbose=1):
    for term in google_search_term_variants:
        try:
            df = scrape_jobs(
                site_name=["google"],
                google_search_term=term,
                results_wanted=results_wanted,
                proxies=proxies,
                verbose=verbose,
            )
        except Exception as e:
            print(f"  [google] google_search_term={term!r} -> raised {type(e).__name__}: {e}")
            continue
        if df is not None and not df.empty:
            print(f"  [google] SUCCESS with google_search_term={term!r} -> {len(df)} jobs")
            return df, term
        else:
            print(f"  [google] google_search_term={term!r} -> 0 jobs, trying next variant")
    print("  [google] All variants exhausted — no results found.")
    return pd.DataFrame(), None


print("Resolving site: google")
google_df, google_term_used = resolve_google(GOOGLE_SEARCH_TERM_VARIANTS, results_wanted=RESULTS_WANTED, proxies=PROXIES)
results["google"] = {"df": google_df, "location": None, "search_term": google_term_used}
print(f"google: {len(google_df)} jobs (google_search_term={google_term_used!r})")


Resolving site: google


2026-06-30 11:33:34,759 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results


  [google] google_search_term='working student data scientist or analyst jobs in Munich, Germany since yesterday' -> 0 jobs, trying next variant


2026-06-30 11:33:35,097 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results


  [google] google_search_term='Werkstudent data scientist jobs near Munich, Germany' -> 0 jobs, trying next variant


2026-06-30 11:33:35,464 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results


  [google] google_search_term='data analyst working student jobs in Munich Germany' -> 0 jobs, trying next variant
  [google] All variants exhausted — no results found.
google: 0 jobs (google_search_term=None)


## Merge results and guarantee a `description` column

`description` already exists in JobSpy's schema for every site, but how well it's
populated varies: Glassdoor, Indeed, and Google fill it by default; LinkedIn only
fills it when `linkedin_fetch_description=True`; ZipRecruiter doesn't populate it
at all. This step merges every site's results into one DataFrame and makes sure
`description` is present as a real column everywhere — raw text where available,
`NaN` where the site never provides it.


In [24]:
import numpy as np

all_dfs = [info["df"] for info in results.values() if not info["df"].empty]

if all_dfs:
    jobs = pd.concat(all_dfs, ignore_index=True)
else:
    jobs = pd.DataFrame()

if not jobs.empty:
    if "description" not in jobs.columns:
        jobs["description"] = np.nan
    # Normalize: empty strings / None -> NaN, keep real text as-is (raw, unmodified)
    jobs["description"] = jobs["description"].replace(r"^\s*$", np.nan, regex=True)
    jobs["description"] = jobs["description"].where(jobs["description"].notna(), np.nan)

    n_with_desc = jobs["description"].notna().sum()
    print(f"Total jobs: {len(jobs)} | with description: {n_with_desc} | without: {len(jobs) - n_with_desc}")
else:
    print("No jobs found across any site/variant combination.")

Total jobs: 145 | with description: 137 | without: 8


C:\Users\Jojo\AppData\Local\Temp\ipykernel_14212\3806708150.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  jobs = pd.concat(all_dfs, ignore_index=True)


In [25]:
jobs = jobs.loc[:,['id', 'site', 'job_url', 'title', 'company','location', 'date_posted', 'job_type', 'job_level', 
       'job_function', 'emails', 'description', 'company_industry', 'company_url', 'skills']]
jobs

,id,site,job_url,title,company,location,date_posted,job_type,job_level,job_function,emails,description,company_industry,company_url,skills
0,gd-1010182609963,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Künstliche Intelligenz & Digitalis...,Bayern Facility Management,München,2026-06-30,None,None,None,bewerbung@bayernfm.de,**Werden Sie Teil unseres Teams.**\n\n#### **I...,None,https://www.glassdoor.de/Overview/W-EI_IE13353...,None
1,gd-1010179417154,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent AI & LLM Engineering (all genders)...,EXXETA,München,2026-06-26,None,None,None,NaN,München\n\nBei Exxeta fordern wir das traditio...,None,https://www.glassdoor.de/Overview/W-EI_IE40696...,None
2,gd-1010179421431,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,ProjectNGES,citema systems GmbH,München,2026-06-26,None,None,None,NaN,NaN,None,https://www.glassdoor.de/Overview/W-EI_IE23878...,None
3,gd-1010173788155,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent (m/w/d) Digital Analytics & Tracki...,FELD M GmbH,München,2026-06-20,None,None,None,NaN,NaN,None,https://www.glassdoor.de/Overview/W-EI_IE31583...,None
4,gd-1010172466536,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Praktikant / Werkstudent (m/w/d) KI-Assistent ...,CHECK24,München,2026-06-19,None,None,None,anja.herger@check24.de,**Im Überblick**\n----------------\n\nStudent\...,None,https://www.glassdoor.de/Overview/W-EI_IE94874...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,li-4378367376,linkedin,https://www.linkedin.com/jobs/view/4378367376,Senior Consultant (m/w/d) Data Science / Künst...,MID GmbH,"Nuremberg, Bavaria, Germany",2026-02-26,fulltime,mid-senior level,Information Technology,p.dueber@mid.de,**Werde #MIDgestalter:in!**\nTeil von MID zu s...,Software Development,https://de.linkedin.com/company/mid-gmbh,None
141,li-4374245632,linkedin,https://www.linkedin.com/jobs/view/4374245632,Studenten(m/w/d) zur Erstellung einer Abschlus...,BERDING BETON GmbH,"Stein bei Nürnberg, Bavaria, Germany",2026-02-23,fulltime,internship,Other,NaN,**Sumitomo (SHI) Demag ist gemeinsam mit seine...,Mechanical Or Industrial Engineering,https://de.linkedin.com/company/berding-beton-...,None
142,li-4374071613,linkedin,https://www.linkedin.com/jobs/view/4374071613,AI & Machine Learning Engineer (all genders),msg,"Nuremberg, Bavaria, Germany",2026-02-19,fulltime,mid-senior level,Engineering and Information Technology,recruiting@msg.group,"Erinnerst du dich an deinen Kindheitstraum, di...",IT Services and IT Consulting,https://de.linkedin.com/company/msggroup,None
143,li-4370665437,linkedin,https://www.linkedin.com/jobs/view/4370665437,Wissenschaftliche Mitarbeitende mit Promotions...,Fraunhofer FIT,"Bayreuth, Bavaria, Germany",2026-02-10,fulltime,internship,Other,lara.gerlach@fit.fraunhofer.de,Wir vom Fraunhofer FIT sind der exzellente Par...,IT Services and IT Consulting,https://de.linkedin.com/company/fraunhofer-fit,None


## Optional: filter for genuine "working student" roles

Job boards don't always respect search terms strictly — this keeps only rows
where the title or description actually mentions "working student" or
"Werkstudent", in case noise slipped through.


In [26]:
import re

if not jobs.empty:
    mask = jobs.apply(
        lambda row: bool(
            re.search(
                r"working student|werkstudent",
                f"{row.get('title','')} {row.get('description','')}",
                re.IGNORECASE,
            )
        ),
        axis=1,
    )
    filtered = jobs[mask]
    print(f"{len(filtered)} of {len(jobs)} rows match 'working student/Werkstudent'")
    filtered.to_csv("jobs_filtered.csv", quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
    filtered
else:
    print("No jobs to filter.")


121 of 145 rows match 'working student/Werkstudent'
